In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, Concatenate
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow as tf


In [ ]:
# ======= RESNET DEFINITION ======= #

# The ResNet structure is defined here, where:
# Dense(factors) -> creates a fully connected layer with 256 neurons;
# ReLU -> introduces non-linearity;
# BatchNormalization -> stabilizes the training process;
# Dropout -> reduces overfitting by randomly disabling neurons during training

def create_resnet(input_dim,
    num_classes,
    n_blocks=5,
    dropout_pcg=0.25,
    factors=256
):
    inp = Input(shape=(input_dim,))

    out = Dense(factors, activation="relu")(inp)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    levels = [out]

    # K residual blocks
    for _ in range(n_blocks):
        res = Dense(factors, activation="relu")(out)
        res = BatchNormalization()(res)
        res = Dropout(dropout_pcg)(res)

        out = Add()([out, res])

        out = Dense(factors, activation="relu")(out)
        out = BatchNormalization()(out)
        out = Dropout(dropout_pcg)(out)

        levels.append(out)

    # Concatenation
    out = Concatenate()(levels + [inp])

    out = Dense(factors, activation="relu")(out)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    # Classificator
    out = Dense(num_classes, activation="softmax")(out)

    m = Model(inputs=inp, outputs=out)

    m.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.RMSprop(1e-3),
        metrics=["accuracy"]
    )

    return m


In [ ]:
# ======= SUBJECT DATASETS (27 SUBJECTS) ======= #

file_list = [
    r'../PSD_data/PSD_EEG_Subject_1.csv',
    r'../PSD_data/PSD_EEG_Subject_2.csv',
    r'../PSD_data/PSD_EEG_Subject_3.csv',
    r'../PSD_data/PSD_EEG_Subject_4.csv',
    r'../PSD_data/PSD_EEG_Subject_5.csv',
    r'../PSD_data/PSD_EEG_Subject_6.csv',
    r'../PSD_data/PSD_EEG_Subject_7.csv',
    r'../PSD_data/PSD_EEG_Subject_8.csv',
    r'../PSD_data/PSD_EEG_Subject_9.csv',
    r'../PSD_data/PSD_EEG_Subject_10.csv',
    r'../PSD_data/PSD_EEG_Subject_11.csv',
    r'../PSD_data/PSD_EEG_Subject_12.csv',
    r'../PSD_data/PSD_EEG_Subject_13.csv',
    r'../PSD_data/PSD_EEG_Subject_14.csv',
    r'../PSD_data/PSD_EEG_Subject_15.csv',
    r'../PSD_data/PSD_EEG_Subject_16.csv',
    r'../PSD_data/PSD_EEG_Subject_17.csv',
    r'../PSD_data/PSD_EEG_Subject_18.csv',
    r'../PSD_data/PSD_EEG_Subject_19.csv',
    r'../PSD_data/PSD_EEG_Subject_20.csv',
    r'../PSD_data/PSD_EEG_Subject_21.csv',
    r'../PSD_data/PSD_EEG_Subject_22.csv',
    r'../PSD_data/PSD_EEG_Subject_23.csv',
    r'../PSD_data/PSD_EEG_Subject_24.csv',
    r'../PSD_data/PSD_EEG_Subject_25.csv',
    r'../PSD_data/PSD_EEG_Subject_26.csv',
    r'../PSD_data/PSD_EEG_Subject_27.csv'
]

# ======= OUTPUT DIRECTORIES ======= #

results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

best_resnet_dir = "best_model_resnet"
os.makedirs(best_resnet_dir, exist_ok=True)

merged_data_dir = "merged_PSD_dataset"
os.makedirs(merged_data_dir, exist_ok=True)


def print_and_save(text, file):
    print(text)
    file.write(text + "\n")

# Number of subjects to be tested
subject_counts = [27, 20, 15, 13]


In [ ]:
# ======= TRAIN AND TEST (SUBJECT SCALING) ======= #

out_txt_path = os.path.join(results_dir, "RESNET_results.txt")

with open(out_txt_path, "w", encoding="utf-8") as f:

    for N in subject_counts:

        print_and_save("\n==========================================", f)
        print_and_save(f"RESNET - TRAINING WITH {N} SUBJECTS", f)
        print_and_save("==========================================", f)

        # ======= DATASET LOADING ======= #
        # Concatenate the first N subject datasets
        df_list = [pd.read_csv(file) for file in file_list[:N]]
        df = pd.concat(df_list, ignore_index=True)

        # Save the merged dataset
        merged_csv = os.path.join(merged_data_dir, f"PSD_dataset_{N}.csv")
        df.to_csv(merged_csv, index=False)

        # Feature PSD (all columns except Event Id)
        X = df.drop("Event Id", axis=1).astype(float).values
        # Labels (Event Id)
        y = df["Event Id"].values

        input_dim = X.shape[1]

        # Class mapping (labels converted to consecutive indices)
        classes = np.array(sorted(np.unique(y)))
        num_classes = len(classes)
        to_index = {c: i for i, c in enumerate(classes)}

        # ======= TRAIN / TEST SPLIT ======= #
        Xtr, Xte, ytr, yte = train_test_split(
            X, y,
            train_size=0.8,
            shuffle=True,
            random_state=42,
            stratify=y
        )

        # ======= SCALER ======= #
        # Fit ONLY on training set to avoid data leakage
        scaler = MinMaxScaler().fit(Xtr)
        Xtr_s = scaler.transform(Xtr)
        Xte_s = scaler.transform(Xte)

        # ======= OVERSAMPLING ======= #
        # Applied ONLY on training set to avoid data leakage
        df_tr = pd.DataFrame(Xtr_s)
        df_tr["Event Id"] = ytr

        max_n = df_tr["Event Id"].value_counts().max()
        dfs = []
        for cls, sub in df_tr.groupby("Event Id"):
            if len(sub) < max_n:
                sub = resample(sub, replace=True, n_samples=max_n, random_state=42)
            dfs.append(sub)

        df_bal = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
        Xtr_bal = df_bal.drop("Event Id", axis=1).values
        ytr_bal = df_bal["Event Id"].values

        # ======= ONE-HOT ENCODING ======= #
        ytr_cat = to_categorical(
            [to_index[c] for c in ytr_bal],
            num_classes=num_classes
        )
        yte_idx = np.array([to_index[c] for c in yte])

        # ======= TRAINING ======= #
        model = create_resnet(input_dim, num_classes)

        ckpt_path = os.path.join(
            best_resnet_dir,
            f"best_resnet_{N}_subjects.weights.h5"
        )

        ckpt = ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            mode="min",
            verbose=0
        )

        rlr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )

        # 20% of balanced training set used as validation
        model.fit(
            Xtr_bal, ytr_cat,
            validation_split=0.2,
            epochs=150,
            batch_size=64,
            callbacks=[ckpt, rlr],
            verbose=0
        )

        # ======= TEST ======= #
        # Load best weights and perform prediction
        model.load_weights(ckpt_path)
        ypred = model.predict(Xte_s, verbose=0)
        ypred_lab = np.argmax(ypred, axis=1)

        acc = accuracy_score(yte_idx, ypred_lab)
        print_and_save(f"\nTest Accuracy: {acc:.4f}", f)

        print_and_save("\nClassification Report:", f)
        print_and_save(
            classification_report(
                yte_idx,
                ypred_lab,
                target_names=[str(c) for c in classes],
                zero_division=0
            ),
            f
        )

        print_and_save(f"\nSaved weights: {ckpt_path}", f)
